In [1]:
import psycopg2
import pandas as pd

In [2]:
db_params = {
    'host': '194.171.191.226',
    'port': '6379',
    'database': 'postgres',
    'user': 'group15',
    'password': 'blockd_2024group15_73'
}

# Connecting to the database

In [3]:
conn_psycopg2 = psycopg2.connect(**db_params)
cursor = conn_psycopg2.cursor()

In [4]:
# Execute SQL query to fetch distinct category types
query = '''
    SELECT * 
    FROM group15_warehouse.accident_data_17_23
'''

cursor.execute(query)

# Fetch all rows
new_accident_rows = cursor.fetchall()

# Fetch column names
column_names = [desc[0] for desc in cursor.description]

# Convert the fetched data into a pandas DataFrame
new_accidents = pd.DataFrame(new_accident_rows, columns=column_names)

display(new_accidents)

,Year,Accident severity,municipality,town,First Mode of Transport,Second mode of Transport,Area Type,Light condition,Road Location,Road condition,Road surface,Road situation,Speed limit,street,weather,accidents
0,2017,Fatal,Breda,BREDA,Car,Pedestrian,Urban area,Darkness,Intersection,Wet/damp,Brick,Bend,30 km/h,Valkeniersplein,Rain,1
1,2017,Fatal,Breda,BREDA,Lorry,Other,Urban area,Daylight,Intersection,Wet/damp,Brick,Intersection - 4 arms,50 km/h,Markendaalseweg,Dry,1
2,2017,Fatal,Breda,BREDA,Lorry,Other,Urban area,Daylight,Road section,Dry,Asphalt (other),Straight road,50 km/h,Academiesingel,Dry,1
3,2017,Injured,Breda,BAVEL,Car,Lorry,Rural area,Darkness,Road section,Wet/damp,Asphalt (other),Bend,120 km/h,KP ST.ANNABOSCH,Dry,1
4,2017,Injured,Breda,BAVEL,Car,Other,Rural area,Darkness,Road section,Wet/damp,Porous asphalt,Straight road,130 km/h,RYKSWG,Rain,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6897,2023,Fatal,Breda,BREDA,Car,Moped,Urban area,Daylight,Road section,Dry,Asphalt (other),Straight road,50 km/h,Terheijdenseweg,Dry,1
6898,2023,Fatal,Breda,BREDA,Car,Car,Urban area,Darkness,Intersection,Dry,Asphalt (other),Intersection - 4 arms,70 km/h,Rijsbergseweg,Dry,1
6899,2023,Fatal,Breda,BREDA,Other,Other,Rural area,Daylight,Road section,Wet/damp,Porous asphalt,Straight road,100 km/h,RYKSWG,Dry,1
6900,2023,Fatal,Breda,PRINSENBEEK,Car,Car,Rural area,Darkness,Road section,Dry,Porous asphalt,Straight road,130 km/h,RYKSWG,Dry,1


## Preprocessing steps

In [5]:
# Step 1: Handle Missing Values
# Check for missing values
missing_values = new_accidents.isnull().sum()
print("Missing values:")
print(missing_values)

Missing values:
Year                        0
Accident severity           0
municipality                0
town                        0
First Mode of Transport     0
Second mode of Transport    0
Area Type                   0
Light condition             0
Road Location               0
Road condition              0
Road surface                0
Road situation              0
Speed limit                 0
street                      0
weather                     0
accidents                   0
dtype: int64


In [6]:
# Step 2: Removing Duplicates
new_accidents.drop_duplicates(inplace=True)

In [7]:
# Step 3: Drop unnecessary columns
# Drop the 'municipality' column
new_accidents = new_accidents.drop(columns=['municipality'])

In [8]:
# Step 4: Make changes in variable values
# Replace 'Other vehicle' and '-' with 'Other' in 'First Mode of Transport'
new_accidents['First Mode of Transport'].replace({'Other vehicle': 'Other', '-': 'Other'})

# Replace 'Other vehicle' with 'Other' in 'Second mode of Transport'
new_accidents['Second mode of Transport'].replace({'Other vehicle': 'Other'})

0       Pedestrian
1            Other
2            Other
3            Lorry
4            Other
           ...    
6897         Moped
6898           Car
6899         Other
6900           Car
6901       Bicycle
Name: Second mode of Transport, Length: 6902, dtype: object

In [9]:
# Step 5: Converting Categorical Variables
# Convert categorical variables to category type
new_accidents['Initial transport'] = new_accidents['First Mode of Transport']
new_accidents['Secondary transport'] = new_accidents['Second mode of Transport']
new_accidents = new_accidents.drop(columns=['First Mode of Transport', 'Second mode of Transport'])

new_accidents.columns = new_accidents.columns.str.capitalize()


categorical_cols = ['Accident severity', 'Town', 'Initial transport', 'Secondary transport',
                    'Area type', 'Light condition', 'Road location', 'Road condition', 'Road surface', 'Road situation', 'Speed limit', 'Street', 'Weather']

for col in categorical_cols:
    new_accidents[col] = new_accidents[col].astype('category')

In [10]:
# Check the datatypes again
print("\nData Types:")
print(new_accidents.info())


Data Types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6902 entries, 0 to 6901
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   Year                 6902 non-null   int64   
 1   Accident severity    6902 non-null   category
 2   Town                 6902 non-null   category
 3   Area type            6902 non-null   category
 4   Light condition      6902 non-null   category
 5   Road location        6902 non-null   category
 6   Road condition       6902 non-null   category
 7   Road surface         6902 non-null   category
 8   Road situation       6902 non-null   category
 9   Speed limit          6902 non-null   category
 10  Street               6902 non-null   category
 11  Weather              6902 non-null   category
 12  Accidents            6902 non-null   int64   
 13  Initial transport    6902 non-null   category
 14  Secondary transport  6902 non-null   category
dtypes: categ

In [11]:
# Step 6: Label encode categorical variables
# Label encode categorical columns
from sklearn.preprocessing import LabelEncoder
label_encoders = {}
for column in categorical_cols:
    le = LabelEncoder()
    new_accidents[column] = le.fit_transform(new_accidents[column])
    label_encoders[column] = le  # Store the label encoder for inverse transformation if needed

In [12]:
# Check the resulting DataFrame
print(new_accidents.head())

   Year  Accident severity  Town  Area type  Light condition  Road location  \
0  2017                  0     1          2                0              0   
1  2017                  0     1          2                1              0   
2  2017                  0     1          2                1              1   
3  2017                  1     0          0                0              1   
4  2017                  1     0          0                0              1   

   Road condition  Road surface  Road situation  Speed limit  Street  Weather  \
0               3             1               0            4     751        3   
1               3             1               2            5     451        0   
2               0             0               4            5       8        0   
3               3             0               0            1     337        0   
4               3             3               4            2     615        3   

   Accidents  Initial transport  Secon

# Modelling

In [13]:
from sklearn.model_selection import train_test_split

# Define feature columns and target variable
X = new_accidents.drop(columns=['Accident severity', 'Town', 'Accidents', 'Year'])
y = new_accidents['Accident severity']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [14]:
from sklearn.ensemble import GradientBoostingClassifier

# Initialize the model
model = GradientBoostingClassifier(n_estimators=100, random_state=42)

# Train the model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

Saving the model

In [15]:
import joblib

joblib.dump(model, 'model.joblib')
print('model saved')

model saved


In [16]:
print(X_train.columns)

Index(['Area type', 'Light condition', 'Road location', 'Road condition',
       'Road surface', 'Road situation', 'Speed limit', 'Street', 'Weather',
       'Initial transport', 'Secondary transport'],
      dtype='object')


In [18]:
le = label_encoders['Accident severity']
print(dict(enumerate(le.classes_)))

{0: 'Fatal', 1: 'Injured', 2: 'Material Damage Only'}
